# NB08 · Consolidación final y paquete editorial

**Entorno recomendado:** CPU, sin acelerador  
**Tiempo estimado:** 3–6 min

Este notebook cierra el flujo del proyecto de clasificación de madurez de palma africana. No entrena ni vuelve a evaluar modelos. Sus funciones son:

1. localizar las últimas ejecuciones completas de NB05, NB06 y NB07;
2. integrar los resultados del benchmark multisemilla de NB04;
3. reconstruir las figuras comparativas principales a 300 dpi;
4. copiar y catalogar las figuras seleccionadas para el libro;
5. reunir todas las tablas en un archivo Excel;
6. generar borradores de métodos, resultados y leyendas;
7. crear un archivo ZIP con el paquete editorial completo.

**Fuente de datos:** *Outdoor Tenera Oil Palm Fruit Image: FFB Dataset*. Zenodo, registro 11114885. DOI: https://doi.org/10.5281/zenodo.11114885

> Los textos generados se basan en los archivos producidos por los notebooks anteriores. Constituyen un borrador académico y deben revisarse antes de incorporarlos al libro o a un manuscrito.


In [ ]:
%pip install -q openpyxl seaborn


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import math
import platform
import shutil
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import openpyxl
import pandas as pd
import seaborn as sns
from PIL import Image, ImageOps

PROJECT_ROOT = Path('/content/drive/MyDrive/PALMA_AFRICANA_YOLO')
DATA_ROOT = PROJECT_ROOT / '02_DATOS_PROCESADOS' / 'classification_5class_v1'
RESULTS_ROOT = PROJECT_ROOT / '06_RESULTADOS'
BOOK_ROOT = PROJECT_ROOT / '08_MATERIAL_LIBRO'

NB04_RESULTS = RESULTS_ROOT / 'NB04_COMPARACION_YOLO11_MULTISEED'
NB05_RESULTS_ROOT = RESULTS_ROOT / 'NB05_DEMOSTRADOR_INFERENCIA'
NB06_RESULTS_ROOT = RESULTS_ROOT / 'NB06_EXPLICABILIDAD_GRADCAM'
NB07_RESULTS_ROOT = RESULTS_ROOT / 'NB07_ROBUSTEZ_PERTURBACIONES'

NB05_BOOK_ROOT = BOOK_ROOT / 'NB05_DEMOSTRADOR_INFERENCIA'
NB06_BOOK_ROOT = BOOK_ROOT / 'NB06_EXPLICABILIDAD_GRADCAM'
NB07_BOOK_ROOT = BOOK_ROOT / 'NB07_ROBUSTEZ_PERTURBACIONES'

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
NB08_RESULTS = RESULTS_ROOT / 'NB08_CONSOLIDACION_EDITORIAL' / RUN_ID
NB08_BOOK = BOOK_ROOT / 'NB08_PAQUETE_FINAL' / RUN_ID
PACKAGE_DIR = NB08_BOOK / 'PAQUETE_EDITORIAL'
FIGURES_DIR = PACKAGE_DIR / '01_FIGURAS'
TABLES_DIR = PACKAGE_DIR / '02_TABLAS'
TEXTS_DIR = PACKAGE_DIR / '03_TEXTOS'
MANIFESTS_DIR = PACKAGE_DIR / '04_MANIFIESTOS'

for path in [NB08_RESULTS, FIGURES_DIR, TABLES_DIR, TEXTS_DIR, MANIFESTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

CLASSES = ['damaged', 'empty', 'overripe', 'ripe', 'unripe']
SPANISH = {
    'damaged': 'dañado',
    'empty': 'vacío',
    'overripe': 'sobremaduro',
    'ripe': 'maduro',
    'unripe': 'inmaduro',
}

print('Entorno: CPU')
print('Paquete editorial:', PACKAGE_DIR)


## 1. Localización y verificación de resultados

Se selecciona la ejecución más reciente que contenga el archivo requerido. Así se evitan carpetas incompletas originadas por ejecuciones interrumpidas.


In [ ]:
def latest_complete_run(root, required_relative_path):
    if not root.exists():
        raise FileNotFoundError(f'No existe la carpeta: {root}')
    candidates = sorted([p for p in root.iterdir() if p.is_dir()], reverse=True)
    for candidate in candidates:
        if (candidate / required_relative_path).exists():
            return candidate
    raise FileNotFoundError(
        f'No se encontró una ejecución completa en {root} con {required_relative_path}'
    )


required_nb04 = [
    NB04_RESULTS / 'resultados_por_corrida.csv',
    NB04_RESULTS / 'resumen_por_modelo.csv',
    NB04_RESULTS / 'seleccion_modelo.json',
    NB04_RESULTS / 'f1_test_por_clase_promedio.csv',
]
missing_nb04 = [str(path) for path in required_nb04 if not path.exists()]
if missing_nb04:
    raise FileNotFoundError('Faltan archivos de NB04:\n' + '\n'.join(missing_nb04))

nb05_run = latest_complete_run(NB05_RESULTS_ROOT, 'predicciones_imagenes_nuevas.csv')
nb06_run = latest_complete_run(NB06_RESULTS_ROOT, 'resumen_gradcam.csv')
nb07_run = latest_complete_run(NB07_RESULTS_ROOT, 'metricas_robustez.csv')

nb05_book_run = NB05_BOOK_ROOT / nb05_run.name
nb06_book_run = NB06_BOOK_ROOT / nb06_run.name
nb07_book_run = NB07_BOOK_ROOT / nb07_run.name

for label, path in {
    'NB05 resultados': nb05_run,
    'NB06 resultados': nb06_run,
    'NB07 resultados': nb07_run,
    'NB05 libro': nb05_book_run,
    'NB06 libro': nb06_book_run,
    'NB07 libro': nb07_book_run,
}.items():
    if not path.exists():
        raise FileNotFoundError(f'No se encontró {label}: {path}')
    print(f'{label}: {path}')


## 2. Carga de tablas y controles de coherencia

La arquitectura final debe coincidir con el modelo empleado en NB05–NB07. La selección se verifica a partir del F1 macro promedio de validación, no del conjunto de prueba.


In [ ]:
nb04_runs = pd.read_csv(NB04_RESULTS / 'resultados_por_corrida.csv')
nb04_models = pd.read_csv(NB04_RESULTS / 'resumen_por_modelo.csv')
nb04_class_f1 = pd.read_csv(NB04_RESULTS / 'f1_test_por_clase_promedio.csv', index_col=0)
selection = json.loads((NB04_RESULTS / 'seleccion_modelo.json').read_text(encoding='utf-8'))

nb05_predictions = pd.read_csv(nb05_run / 'predicciones_imagenes_nuevas.csv')
nb06_gradcam = pd.read_csv(nb06_run / 'resumen_gradcam.csv')
nb07_metrics = pd.read_csv(nb07_run / 'metricas_robustez.csv')
nb07_statistics = pd.read_csv(nb07_run / 'comparaciones_pareadas.csv')
nb07_per_class = pd.read_csv(nb07_run / 'recall_por_clase_y_condicion.csv')

selected_model = str(selection['selected_model'])
if selected_model != 'yolo11m':
    raise RuntimeError(
        f'NB04 seleccionó {selected_model}, pero NB05–NB07 usan YOLO11m. '
        'No se generará un paquete editorial inconsistente.'
    )

if set(nb04_runs['seed'].astype(int)) != {2026, 2027, 2028}:
    raise RuntimeError('NB04 no contiene exactamente las semillas 2026, 2027 y 2028.')
expected_pairs = {
    (model, seed)
    for model in ['yolo11n', 'yolo11s', 'yolo11m']
    for seed in [2026, 2027, 2028]
}
observed_pairs = set(zip(nb04_runs['model_key'], nb04_runs['seed'].astype(int)))
if observed_pairs != expected_pairs or len(nb04_runs) != 9:
    missing_pairs = sorted(expected_pairs - observed_pairs)
    extra_pairs = sorted(observed_pairs - expected_pairs)
    raise RuntimeError(
        f'NB04 debe contener 9 corridas (3 modelos × 3 semillas). '
        f'Faltantes: {missing_pairs}; adicionales: {extra_pairs}; filas: {len(nb04_runs)}.'
    )
if 'clean' not in set(nb07_metrics['id']):
    raise RuntimeError('NB07 no contiene la condición original de referencia.')

runs_per_model = nb04_runs.groupby('model_key').agg(
    n_corridas=('seed', 'count'),
    semillas=('seed', lambda values: ', '.join(map(str, sorted(values.astype(int))))),
)
winner_seeds = set(
    nb04_runs.loc[nb04_runs['model_key'] == selected_model, 'seed'].astype(int)
)
if winner_seeds != {2026, 2027, 2028}:
    raise RuntimeError(
        f'El modelo final {selected_model} debe conservar las tres semillas; '
        f'se encontraron {sorted(winner_seeds)}.'
    )
runs_per_model_text = '; '.join(
    f"{model}: {int(row['n_corridas'])} corridas "
    f"(semillas {row['semillas']})"
    for model, row in runs_per_model.iterrows()
)

display(nb04_models.round(4))
display(runs_per_model)
display(nb07_metrics[[
    'label', 'macro_f1', 'macro_f1_drop', 'stable_label_rate'
]].round(4))
print('Modelo seleccionado por validación:', selected_model)


In [ ]:
# Inventario físico del conjunto procesado.
split_rows = []
for split in ['train', 'val', 'test']:
    for class_name in CLASSES:
        folder = DATA_ROOT / split / class_name
        if not folder.exists():
            raise FileNotFoundError(f'Falta la carpeta: {folder}')
        n = sum(1 for path in folder.iterdir() if path.is_file())
        split_rows.append({'split': split, 'class': class_name, 'n': n})

split_distribution = pd.DataFrame(split_rows)
split_pivot = split_distribution.pivot(index='class', columns='split', values='n')
expected_totals = {'train': 312, 'val': 70, 'test': 70}
observed_totals = split_distribution.groupby('split')['n'].sum().to_dict()
if observed_totals != expected_totals:
    raise RuntimeError(f'Distribución inesperada: {observed_totals}')

display(split_pivot[['train', 'val', 'test']])
print('Total:', int(split_distribution['n'].sum()))


## 3. Figuras comparativas reconstruidas a 300 dpi

Las dos figuras de NB04 se reconstruyen desde las tablas guardadas para mejorar su resolución y uniformar los nombres de las clases.


In [ ]:
sns.set_theme(style='whitegrid', context='notebook')
MODEL_LABELS = {'yolo11n': 'YOLO11n', 'yolo11s': 'YOLO11s', 'yolo11m': 'YOLO11m'}

plot_runs = nb04_runs.copy()
plot_runs['Modelo'] = plot_runs['model_key'].map(MODEL_LABELS).fillna(plot_runs['model_key'])
model_order = [
    MODEL_LABELS.get(model, model)
    for model in nb04_models.sort_values('validation_rank')['model_key']
]

# Figura editorial en dos paneles. Las cajas facilitan comparar la dispersión;
# los puntos muestran las tres semillas y el diamante indica la media.
palette = {
    'validacion_box': '#A9CBE0',
    'validacion_point': '#155A8A',
    'prueba_box': '#B9D8B4',
    'prueba_point': '#2E6B3C',
}

fig, axes = plt.subplots(
    1, 2,
    figsize=(9.4, 4.4),
    sharey=True,
    constrained_layout=True,
)
fig.patch.set_facecolor('white')

panel_specs = [
    (axes[0], 'val_f1_macro', 'a) Validación', palette['validacion_box'], palette['validacion_point']),
    (axes[1], 'test_f1_macro', 'b) Prueba', palette['prueba_box'], palette['prueba_point']),
]

for ax, metric, title, box_color, point_color in panel_specs:
    sns.boxplot(
        data=plot_runs,
        x='Modelo',
        y=metric,
        order=model_order,
        width=0.48,
        showfliers=False,
        color=box_color,
        linewidth=1.25,
        boxprops={'edgecolor': '#52606D'},
        whiskerprops={'color': '#52606D', 'linewidth': 1.15},
        capprops={'color': '#52606D', 'linewidth': 1.15},
        medianprops={'color': '#1F2933', 'linewidth': 1.6},
        ax=ax,
    )
    sns.stripplot(
        data=plot_runs,
        x='Modelo',
        y=metric,
        order=model_order,
        color=point_color,
        size=5.8,
        jitter=0.075,
        alpha=0.92,
        edgecolor='white',
        linewidth=0.55,
        zorder=4,
        ax=ax,
    )

    means = plot_runs.groupby('Modelo')[metric].mean().reindex(model_order)
    ax.scatter(
        np.arange(len(model_order)),
        means.to_numpy(),
        marker='D',
        s=44,
        color='#F4A261',
        edgecolor='white',
        linewidth=0.8,
        zorder=5,
    )
    maxima = plot_runs.groupby('Modelo')[metric].max().reindex(model_order)
    for x_position, (mean_value, maximum_value) in enumerate(zip(means, maxima)):
        ax.text(
            x_position,
            min(0.992, maximum_value + 0.006),
            f'μ = {mean_value:.3f}',
            ha='center',
            va='bottom',
            fontsize=8.0,
            color='#334E68',
            fontweight='semibold',
        )

    ax.set_title(title, fontsize=11.5, fontweight='semibold', pad=8)
    ax.set_xlabel('Arquitectura', fontsize=10)
    ax.set_ylim(0.75, 1.00)
    ax.set_yticks(np.arange(0.75, 1.001, 0.05))
    ax.grid(axis='y', color='#D9E2EC', linewidth=0.75, alpha=0.8)
    ax.grid(axis='x', visible=False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#9AA5B1')
    ax.spines['bottom'].set_color('#9AA5B1')
    ax.tick_params(axis='both', labelsize=9.5)

axes[0].set_ylabel('F1 macro', fontsize=10.5)
axes[1].set_ylabel('')

# Identifica el modelo seleccionado sin cubrir los datos.
selected_label = MODEL_LABELS.get(selected_model, selected_model)
selected_position = model_order.index(selected_label)
axes[0].get_xticklabels()[selected_position].set_fontweight('bold')
axes[0].get_xticklabels()[selected_position].set_color('#155A8A')
axes[1].get_xticklabels()[selected_position].set_fontweight('bold')
axes[1].get_xticklabels()[selected_position].set_color('#155A8A')

benchmark_figure = FIGURES_DIR / 'F01_benchmark_f1_macro_multisemilla.png'
fig.savefig(
    benchmark_figure, dpi=300, bbox_inches='tight', pad_inches=0.08, facecolor='white'
)
plt.show()

class_f1_plot = nb04_class_f1.copy()
class_f1_plot.index = [MODEL_LABELS.get(str(idx), str(idx)) for idx in class_f1_plot.index]
class_f1_plot.columns = [SPANISH.get(str(col), str(col)).capitalize() for col in class_f1_plot.columns]
class_f1_plot = class_f1_plot.reindex(model_order)

fig, ax = plt.subplots(figsize=(9.2, 4.5), constrained_layout=True)
sns.heatmap(
    class_f1_plot, annot=True, fmt='.3f', cmap='YlGnBu', vmin=0, vmax=1,
    linewidths=0.5, cbar_kws={'label': 'F1 promedio'}, ax=ax
)
ax.set_title('F1 por clase en prueba · promedio de tres semillas', fontweight='bold')
ax.set_xlabel('Clase')
ax.set_ylabel('Arquitectura')
class_figure = FIGURES_DIR / 'F02_f1_promedio_por_clase.png'
fig.savefig(
    class_figure, dpi=300, bbox_inches='tight', pad_inches=0.08, facecolor='white'
)
plt.show()


## 4. Selección y copia de figuras

Se conserva una figura individual de inferencia, una explicación Grad-CAM y las cuatro figuras de robustez. Los archivos originales permanecen en sus carpetas; NB08 trabaja con copias estandarizadas.


In [ ]:
def first_png(folder):
    candidates = sorted(folder.glob('*.png'))
    if not candidates:
        raise FileNotFoundError(f'No se encontraron PNG en {folder}')
    return candidates[0]


figure_sources = [
    (first_png(nb05_book_run / 'figuras_individuales'), 'F03_inferencia_imagen_nueva.png', 'NB05'),
    (first_png(nb06_book_run / 'figuras_gradcam'), 'F04_explicabilidad_gradcam.png', 'NB06'),
    (nb07_book_run / 'figuras' / 'perdida_f1_macro_intervalos.png', 'F05_perdida_f1_robustez.png', 'NB07'),
    (nb07_book_run / 'figuras' / 'curvas_robustez_por_familia.png', 'F06_curvas_robustez.png', 'NB07'),
    (nb07_book_run / 'figuras' / 'ejemplo_perturbaciones_controladas.png', 'F07_ejemplo_perturbaciones.png', 'NB07'),
    (nb07_book_run / 'figuras' / 'matriz_confusion_peor_condicion.png', 'F08_matriz_confusion_peor_condicion.png', 'NB07'),
]

copied_figures = []
for source, destination_name, notebook_source in figure_sources:
    if not source.exists():
        raise FileNotFoundError(f'No se encontró la figura requerida: {source}')
    destination = FIGURES_DIR / destination_name
    shutil.copy2(source, destination)
    copied_figures.append((destination, notebook_source, source))
    print('Copiada:', destination_name)


In [ ]:
# Catálogo técnico de figuras.
all_figures = sorted(FIGURES_DIR.glob('*.png'))
catalog_rows = []
for figure_path in all_figures:
    with Image.open(figure_path) as image:
        width, height = image.size
        dpi = image.info.get('dpi', (None, None))
    source_record = next(
        ((notebook, str(source)) for path, notebook, source in copied_figures if path == figure_path),
        ('NB08', 'Reconstruida desde tablas'),
    )
    catalog_rows.append({
        'archivo': figure_path.name,
        'origen': source_record[0],
        'fuente_original': source_record[1],
        'ancho_px': width,
        'alto_px': height,
        'dpi_x_metadato': None if dpi[0] is None else round(float(dpi[0]), 2),
        'dpi_y_metadato': None if dpi[1] is None else round(float(dpi[1]), 2),
        'ancho_maximo_cm_a_300dpi': round(width / 300 * 2.54, 2),
        'tamano_mb': round(figure_path.stat().st_size / 1024**2, 3),
    })

figure_catalog = pd.DataFrame(catalog_rows)
figure_catalog.to_csv(MANIFESTS_DIR / 'catalogo_figuras.csv', index=False)
display(figure_catalog)


## 5. Tabla maestra y libro Excel

El libro Excel conserva los resultados originales en hojas separadas. No se redondean los archivos fuente; el redondeo se limita a las visualizaciones del notebook.


In [ ]:
winner = nb04_models.sort_values('validation_rank').iloc[0]
clean_robustness = nb07_metrics.loc[nb07_metrics['id'] == 'clean'].iloc[0]
worst_robustness = nb07_metrics.query("id != 'clean'").sort_values('macro_f1').iloc[0]
best_perturbed = nb07_metrics.query("id != 'clean'").sort_values('macro_f1', ascending=False).iloc[0]

key_metrics = pd.DataFrame([
    {'grupo': 'Datos', 'metrica': 'Imágenes totales', 'valor': int(split_distribution['n'].sum()), 'detalle': '312 entrenamiento, 70 validación y 70 prueba'},
    {'grupo': 'Benchmark', 'metrica': 'Modelo seleccionado', 'valor': selected_model, 'detalle': selection['selection_rule']},
    {'grupo': 'Benchmark', 'metrica': 'F1 macro validación promedio', 'valor': float(winner['val_f1_macro_mean']), 'detalle': f"DE = {winner['val_f1_macro_std']:.4f}; 3 semillas del modelo final"},
    {'grupo': 'Benchmark', 'metrica': 'F1 macro prueba promedio', 'valor': float(winner['test_f1_macro_mean']), 'detalle': f"DE = {winner['test_f1_macro_std']:.4f}; 3 semillas del modelo final"},
    {'grupo': 'Eficiencia', 'metrica': 'Tamaño del modelo (MB)', 'valor': float(winner['model_size_mb_mean']), 'detalle': 'Promedio de las tres semillas del modelo final'},
    {'grupo': 'Eficiencia', 'metrica': 'Inferencia (ms/imagen)', 'valor': float(winner['inference_ms_image_mean']), 'detalle': 'Promedio en prueba'},
    {'grupo': 'Robustez', 'metrica': 'F1 macro original', 'valor': float(clean_robustness['macro_f1']), 'detalle': 'YOLO11m, semilla 2026'},
    {'grupo': 'Robustez', 'metrica': 'Condición más exigente', 'valor': worst_robustness['label'], 'detalle': f"F1 macro = {worst_robustness['macro_f1']:.4f}"},
    {'grupo': 'Robustez', 'metrica': 'Mayor F1 entre perturbaciones', 'valor': best_perturbed['label'], 'detalle': f"F1 macro = {best_perturbed['macro_f1']:.4f}"},
    {'grupo': 'Demostrador', 'metrica': 'Imágenes nuevas analizadas', 'valor': len(nb05_predictions), 'detalle': 'NB05'},
    {'grupo': 'Explicabilidad', 'metrica': 'Imágenes explicadas', 'valor': len(nb06_gradcam), 'detalle': 'Grad-CAM, NB06'},
])
key_metrics.to_csv(TABLES_DIR / 'tabla_maestra_resultados.csv', index=False)
split_distribution.to_csv(TABLES_DIR / 'distribucion_dataset.csv', index=False)

excel_path = TABLES_DIR / 'Resultados_Palma_Africana_NB04_NB07.xlsx'
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    key_metrics.to_excel(writer, sheet_name='metricas_clave', index=False)
    split_distribution.to_excel(writer, sheet_name='distribucion_datos', index=False)
    nb04_runs.to_excel(writer, sheet_name='corridas_NB04', index=False)
    nb04_models.to_excel(writer, sheet_name='modelos_NB04', index=False)
    nb04_class_f1.reset_index().to_excel(writer, sheet_name='f1_clases_NB04', index=False)
    nb05_predictions.to_excel(writer, sheet_name='predicciones_NB05', index=False)
    nb06_gradcam.to_excel(writer, sheet_name='gradcam_NB06', index=False)
    nb07_metrics.to_excel(writer, sheet_name='robustez_NB07', index=False)
    nb07_statistics.to_excel(writer, sheet_name='estadistica_NB07', index=False)
    nb07_per_class.to_excel(writer, sheet_name='clases_NB07', index=False)
    figure_catalog.to_excel(writer, sheet_name='catalogo_figuras', index=False)

workbook = openpyxl.load_workbook(excel_path)
for worksheet in workbook.worksheets:
    worksheet.freeze_panes = 'A2'
    worksheet.auto_filter.ref = worksheet.dimensions
    for cell in worksheet[1]:
        cell.font = openpyxl.styles.Font(bold=True, color='FFFFFF')
        cell.fill = openpyxl.styles.PatternFill('solid', fgColor='1F4E78')
    for column_cells in worksheet.columns:
        width = min(45, max(10, max(len(str(cell.value or '')) for cell in column_cells) + 2))
        worksheet.column_dimensions[column_cells[0].column_letter].width = width
workbook.save(excel_path)

display(key_metrics)
print('Libro Excel:', excel_path)


## 6. Borradores académicos y leyendas

Los textos se generan con las cifras observadas. Las expresiones “puntaje” y “asociación espacial” se mantienen deliberadamente: los puntajes del modelo no se presentan como probabilidades calibradas y Grad-CAM no demuestra causalidad.


In [ ]:
winner_label = MODEL_LABELS.get(selected_model, selected_model)
worst_stats = nb07_statistics.loc[
    nb07_statistics['condition_id'] == worst_robustness['id']
].iloc[0]

methods_text = f'''# Borrador de métodos

Se utilizó el conjunto público *Outdoor Tenera Oil Palm Fruit Image: FFB Dataset* (Zenodo 11114885; DOI: 10.5281/zenodo.11114885). Después de la auditoría de integridad y la agrupación de duplicados exactos y visualmente próximos, se construyó una partición sin cruce de grupos compuesta por 312 imágenes de entrenamiento, 70 de validación y 70 de prueba. Se completaron nueve corridas multisemilla entre YOLO11n, YOLO11s y YOLO11m ({runs_per_model_text}), correspondientes a las semillas 2026, 2027 y 2028 para cada arquitectura. La selección se realizó exclusivamente con el F1 macro promedio de validación.

El modelo seleccionado, {winner_label}, se examinó mediante un demostrador de inferencia, explicaciones Grad-CAM y un análisis de robustez pareado. La robustez se evaluó sobre las 70 imágenes de prueba mediante versiones simuladas con oscurecimiento, sobreexposición, reducción de contraste, desenfoque, ruido, rotación y compresión JPEG, cada una con intensidades moderada y severa. Se estimaron intervalos de confianza del 95 % para la pérdida de F1 macro mediante 2.000 remuestreos bootstrap pareados. Los cambios en la proporción de aciertos se contrastaron con la prueba exacta de McNemar y corrección de Holm.
'''

results_text = f'''# Borrador de resultados

De acuerdo con el criterio preespecificado, {winner_label} ocupó el primer lugar en validación, con un F1 macro promedio de {winner['val_f1_macro_mean']:.3f} ± {winner['val_f1_macro_std']:.3f} en tres semillas. En el conjunto de prueba alcanzó un F1 macro promedio de {winner['test_f1_macro_mean']:.3f} ± {winner['test_f1_macro_std']:.3f}. El tamaño promedio del modelo fue de {winner['model_size_mb_mean']:.2f} MB y el tiempo medio de inferencia fue de {winner['inference_ms_image_mean']:.2f} ms por imagen en el entorno registrado por NB04.

En el análisis de robustez del modelo YOLO11m con semilla 2026, la condición original obtuvo un F1 macro de {clean_robustness['macro_f1']:.3f}. La condición con menor desempeño fue **{worst_robustness['label']}**, con F1 macro de {worst_robustness['macro_f1']:.3f}, pérdida absoluta de {worst_robustness['macro_f1_drop']:.3f} y estabilidad de etiqueta de {worst_robustness['stable_label_rate']:.1%}. El intervalo bootstrap del 95 % para la pérdida de F1 macro fue [{worst_stats['macro_f1_drop_ci95_low']:.3f}, {worst_stats['macro_f1_drop_ci95_high']:.3f}], y el valor *p* de McNemar corregido por Holm fue {worst_stats['mcnemar_p_holm']:.4g}.

Las visualizaciones Grad-CAM se utilizaron para identificar regiones espacialmente asociadas con cada predicción. Estas visualizaciones no demuestran causalidad. Asimismo, las perturbaciones son simuladas sobre las mismas 70 imágenes y no sustituyen una validación externa con fotografías nuevas de otros lotes, dispositivos y condiciones ambientales.
'''

captions_text = '''# Leyendas propuestas

**Figura 1.** Comparación multisemilla de las arquitecturas YOLO11 en validación y prueba. Las cajas representan el rango intercuartílico, la línea central corresponde a la mediana, los puntos muestran las semillas 2026, 2027 y 2028, y el diamante indica la media. YOLO11m se seleccionó exclusivamente con el conjunto de validación.

**Figura 2.** F1 promedio por clase en el conjunto de prueba para cada arquitectura YOLO11. Los valores corresponden al promedio de tres semillas.

**Figura 3.** Ejemplo de clasificación de madurez en una fotografía nueva. Se presenta la imagen analizada y la distribución de puntajes entre las cinco clases.

**Figura 4.** Explicación Grad-CAM de una predicción. Se muestran la fotografía original, la entrada cuadrada del modelo, el mapa de activación y su superposición. Los colores cálidos indican mayor asociación espacial con la clase predicha.

**Figura 5.** Pérdida de F1 macro frente a perturbaciones controladas. Las barras de error representan intervalos de confianza del 95 % obtenidos mediante bootstrap pareado.

**Figura 6.** Degradación del F1 macro entre la condición original y las intensidades moderada y severa de cada familia de perturbación.

**Figura 7.** Ejemplo visual de las perturbaciones simuladas empleadas para examinar la sensibilidad del clasificador.

**Figura 8.** Matriz de confusión normalizada bajo la condición que produjo el menor F1 macro.
'''

(TEXTS_DIR / '01_borrador_metodos.md').write_text(methods_text, encoding='utf-8')
(TEXTS_DIR / '02_borrador_resultados.md').write_text(results_text, encoding='utf-8')
(TEXTS_DIR / '03_leyendas_figuras.md').write_text(captions_text, encoding='utf-8')

print(methods_text)
print(results_text)


## 7. Índice visual y paquete ZIP

El índice visual es una hoja de revisión; no está destinado a insertarse directamente en el libro. El ZIP contiene las figuras, tablas, textos y manifiestos consolidados.


In [ ]:
def preview_image(path, max_size=(900, 650)):
    with Image.open(path) as image:
        preview = ImageOps.exif_transpose(image).convert('RGB')
        preview.thumbnail(max_size, Image.Resampling.LANCZOS)
        return preview.copy()


ncols = 2
nrows = math.ceil(len(all_figures) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4.8 * nrows), constrained_layout=True)
axes = np.asarray(axes).reshape(-1)
for ax in axes:
    ax.axis('off')
for ax, figure_path in zip(axes, all_figures):
    ax.imshow(preview_image(figure_path))
    ax.set_title(figure_path.name, fontsize=10, fontweight='semibold')
    ax.axis('off')
fig.suptitle('Índice visual del paquete editorial', fontsize=16, fontweight='bold')
visual_index = MANIFESTS_DIR / 'indice_visual_figuras.png'
fig.savefig(
    visual_index, dpi=180, bbox_inches='tight', pad_inches=0.08, facecolor='white'
)
plt.show()

source_manifest = {
    'run_id': RUN_ID,
    'dataset': {
        'title': 'Outdoor Tenera Oil Palm Fruit Image: FFB Dataset',
        'zenodo_record': 'https://zenodo.org/records/11114885',
        'doi': 'https://doi.org/10.5281/zenodo.11114885',
    },
    'selected_model': selected_model,
    'selection_rule': selection['selection_rule'],
    'source_runs': {
        'NB04': str(NB04_RESULTS),
        'NB05': str(nb05_run),
        'NB06': str(nb06_run),
        'NB07': str(nb07_run),
    },
    'files_in_package': sorted(
        str(path.relative_to(PACKAGE_DIR))
        for path in PACKAGE_DIR.rglob('*') if path.is_file()
    ),
    'python': sys.version,
    'platform': platform.platform(),
}
(MANIFESTS_DIR / 'manifiesto_paquete.json').write_text(
    json.dumps(source_manifest, ensure_ascii=False, indent=2), encoding='utf-8'
)

readme = f'''# Paquete editorial · Palma africana

Generado por NB08 el {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}.

## Contenido

- `01_FIGURAS`: ocho figuras numeradas para selección editorial.
- `02_TABLAS`: tabla maestra, distribución del dataset y libro Excel.
- `03_TEXTOS`: borradores de métodos, resultados y leyendas.
- `04_MANIFIESTOS`: catálogo técnico, índice visual y trazabilidad.

## Advertencias

- La selección de arquitectura se realizó con validación.
- El conjunto de prueba se reservó para estimación final y análisis descriptivo de robustez.
- Las perturbaciones son simuladas y no constituyen validación externa.
- Grad-CAM muestra asociación espacial, no causalidad.
'''
(PACKAGE_DIR / 'README.md').write_text(readme, encoding='utf-8')

zip_base = NB08_BOOK / f'PAQUETE_EDITORIAL_PALMA_NB08_{RUN_ID}'
zip_path = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=PACKAGE_DIR))

summary = {
    'run_id': RUN_ID,
    'selected_model': selected_model,
    'figures': len(all_figures),
    'excel_file': str(excel_path),
    'package_directory': str(PACKAGE_DIR),
    'zip_file': str(zip_path),
    'source_nb05_run': nb05_run.name,
    'source_nb06_run': nb06_run.name,
    'source_nb07_run': nb07_run.name,
}
(NB08_RESULTS / 'resumen_nb08.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
)

print('NB08 FINALIZADO CORRECTAMENTE')
print('Paquete editorial:', PACKAGE_DIR)
print('Archivo ZIP:', zip_path)


## Resultado esperado

Al finalizar se mostrará el mensaje `NB08 FINALIZADO CORRECTAMENTE`. El paquete quedará almacenado en:

`PALMA_AFRICANA_YOLO/08_MATERIAL_LIBRO/NB08_PAQUETE_FINAL/`

Este paquete permite comenzar la redacción del capítulo aplicado de palma africana sin buscar manualmente resultados en múltiples carpetas.
